# 02 — Data Preprocessing
**E-Waste Toxic Gas Detection System — ML Pipeline**

**Purpose:** Clean, normalize, encode and split the dataset. Outputs ready-to-train arrays.

**Author:** Sanjula Madushanka | Final Year Research Y4S2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR   = Path('../datasets')
SAVE_DIR   = Path('../results')
MODEL_DIR  = Path('../models')
for d in [SAVE_DIR, MODEL_DIR, DATA_DIR / 'processed' / 'train_test_split']:
    d.mkdir(parents=True, exist_ok=True)

print('Libraries loaded ✅')

## 2.1 Load Raw Data

In [ ]:
# Load primary dataset
df = pd.read_csv(DATA_DIR / 'collected' / 'lab_readings.csv')
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2.2 Standardise Column Names

In [ ]:
# Expected column names — rename if dataset uses different names
REQUIRED_FEATURES = ['mq2_ppm', 'mq7_ppm', 'mq135_ppm', 'mq303_ppm', 'mq136_ppm',
                     'temperature_c', 'humidity_pct']
TARGET_COL = 'gas_class'

# If your dataset has different names, add a mapping here:
COLUMN_MAP = {
    # 'old_name': 'new_name',
    # Example: 'Temp': 'temperature_c',
}
if COLUMN_MAP:
    df = df.rename(columns=COLUMN_MAP)
    print(f'Renamed columns: {COLUMN_MAP}')

# Verify all required columns exist
missing_cols = [c for c in REQUIRED_FEATURES + [TARGET_COL] if c not in df.columns]
if missing_cols:
    print(f'⚠ Missing columns: {missing_cols}')
    print('  → Add them to COLUMN_MAP above or check your dataset')
else:
    print('✅ All required columns present')

df = df[REQUIRED_FEATURES + [TARGET_COL]]
print(f'Final shape: {df.shape}')

## 2.3 Handle Missing Values

In [ ]:
print(f'Missing values before: {df.isnull().sum().sum()}')

# Strategy: median imputation for numeric, mode for categorical
for col in REQUIRED_FEATURES:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Filled {col} with median = {median_val:.4f}')

if df[TARGET_COL].isnull().any():
    df.dropna(subset=[TARGET_COL], inplace=True)
    print(f'  Dropped rows with missing target label')

print(f'Missing values after:  {df.isnull().sum().sum()}')
print(f'Rows remaining: {len(df)}')

## 2.4 Remove Duplicates

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'Removed {before - after} duplicate rows ({before} → {after})')

## 2.5 Remove Outliers (IQR Method)

In [ ]:
before = len(df)
for col in REQUIRED_FEATURES:
    Q1  = df[col].quantile(0.01)   # Use 1st/99th percentile (lenient)
    Q3  = df[col].quantile(0.99)
    IQR = Q3 - Q1
    lower = Q1 - 3.0 * IQR        # Very lenient: 3× IQR (keep more data)
    upper = Q3 + 3.0 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f'Removed {before - after} extreme outlier rows ({before} → {after})')
print('Note: Using 3×IQR threshold to keep realistic sensor noise')

## 2.6 Label Encoding

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[TARGET_COL])

print('=== LABEL ENCODING ===')
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    count = (df['label_encoded'] == idx).sum()
    print(f'  {idx}: {cls:10s} → {count} samples')

# Save label encoder
joblib.dump(le, MODEL_DIR / 'label_encoder.pkl')
print(f'\n✅ LabelEncoder saved to {MODEL_DIR}/label_encoder.pkl')
print(f'   Classes: {list(le.classes_)}')

## 2.7 Feature/Target Split

In [ ]:
X = df[REQUIRED_FEATURES].values
y = df['label_encoded'].values

print(f'Feature matrix X: {X.shape}  (samples × features)')
print(f'Target vector  y: {y.shape}')
print(f'\nFeature names: {REQUIRED_FEATURES}')
print(f'Unique classes: {np.unique(y)} → {list(le.classes_)}')

## 2.8 Train/Test Split (80/20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Ensures each class is proportionally represented
)

print('=== TRAIN/TEST SPLIT (80/20 Stratified) ===')
print(f'Training set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')
print()
print('Training class distribution:')
for cls_idx in np.unique(y_train):
    count = (y_train == cls_idx).sum()
    print(f'  {le.classes_[cls_idx]:10s}: {count}')
print()
print('Test class distribution:')
for cls_idx in np.unique(y_test):
    count = (y_test == cls_idx).sum()
    print(f'  {le.classes_[cls_idx]:10s}: {count}')

## 2.9 Normalize Features (MinMaxScaler)
> **Important:** Fit scaler on TRAINING data only, then transform both train and test to prevent data leakage.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train only
X_test_scaled  = scaler.transform(X_test)         # Transform test (no fit)

print('=== FEATURE SCALING (MinMaxScaler) ===')
print(f'X_train: min={X_train_scaled.min():.4f}, max={X_train_scaled.max():.4f}')
print(f'X_test:  min={X_test_scaled.min():.4f},  max={X_test_scaled.max():.4f}')

# Save scaler
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')
print(f'\n✅ MinMaxScaler saved to {MODEL_DIR}/scaler.pkl')
print('   (This exact scaler must be used during FastAPI inference)')

## 2.10 SMOTE — Handle Class Imbalance (If Needed)

In [ ]:
from collections import Counter

class_counts = Counter(y_train)
imbalance_ratio = max(class_counts.values()) / min(class_counts.values())

print(f'Class imbalance ratio: {imbalance_ratio:.2f}x')

APPLY_SMOTE = imbalance_ratio > 2.0  # Apply if imbalance > 2x

if APPLY_SMOTE:
    print('⚠ Imbalance detected → Applying SMOTE...')
    smote = SMOTE(random_state=42, k_neighbors=min(5, min(class_counts.values()) - 1))
    X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)
    
    print('After SMOTE:')
    new_counts = Counter(y_train_final)
    for cls_idx in sorted(new_counts.keys()):
        print(f'  {le.classes_[cls_idx]:10s}: {new_counts[cls_idx]}')
    print(f'Total training samples: {len(X_train_final)}')
else:
    print('✅ Class distribution acceptable — SMOTE not needed')
    X_train_final, y_train_final = X_train_scaled, y_train
    print(f'Training samples: {len(X_train_final)}')

## 2.11 Save Processed Splits

In [ ]:
SPLIT_DIR = DATA_DIR / 'processed' / 'train_test_split'

pd.DataFrame(X_train_final, columns=REQUIRED_FEATURES).to_csv(SPLIT_DIR / 'X_train.csv', index=False)
pd.DataFrame(X_test_scaled,  columns=REQUIRED_FEATURES).to_csv(SPLIT_DIR / 'X_test.csv',  index=False)
pd.DataFrame(y_train_final,  columns=['label']).to_csv(SPLIT_DIR / 'y_train.csv', index=False)
pd.DataFrame(y_test,          columns=['label']).to_csv(SPLIT_DIR / 'y_test.csv',  index=False)

# Also save combined processed CSV
df.to_csv(DATA_DIR / 'processed' / 'combined_dataset.csv', index=False)

print('=== SAVED FILES ===')
for f in SPLIT_DIR.iterdir():
    print(f'  ✅ {f.name} ({f.stat().st_size / 1024:.1f} KB)')
print(f'  ✅ combined_dataset.csv')
print()
print('PREPROCESSING COMPLETE — proceed to 03_feature_engineering.ipynb')